# 1️⃣ Setup and Split Data

## What This Section Does
We start by loading the credit card fraud dataset, building the feature vector, and splitting the data into training and testing sets.

## Why This Matters
Before comparing balancing methods, we need a clean pipeline that every model can use in the same way. That makes the comparison fair.

## Code Breakdown
The code below:
- Starts a Spark session
- Loads `../data/creditcard.csv`
- Selects the feature columns and removes `Time` and `Class` from the input vector
- Combines all features into a single `features` column with `VectorAssembler`
- Splits the data into training and test sets
- Separates fraud and normal rows inside the training set so we can rebalance later

## What to Notice
- `train_raw` is the original training data
- `test_data` is held out for evaluation only
- `fraud_train` and `normal_train` are used to build balanced versions of the training set

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.mllib.evaluation import MulticlassMetrics

spark = SparkSession.builder.appName("Balancing_Showdown").getOrCreate()

# 1. Load and prepare features
raw_df = spark.read.csv("../data/creditcard.csv", header=True, inferSchema=True)
feature_cols = [c for c in raw_df.columns if c not in ['Class', 'Time']]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
prepared_df = assembler.transform(raw_df).select("features", "Class")

# 2. Split into Train and Test
train_raw, test_data = prepared_df.randomSplit([0.8, 0.2], seed=42)
train_raw.cache() # Speeds up processing since we use this multiple times

# Separate classes for sampling
fraud_train = train_raw.filter(col("Class") == 1)
normal_train = train_raw.filter(col("Class") == 0)

fraud_count = fraud_train.count()
normal_count = normal_train.count()

print(f"Train Normal: {normal_count} | Train Fraud: {fraud_count}")

your 131072x1 screen size is bogus. expect trouble
26/04/29 15:35:35 WARN Utils: Your hostname, SerbansLaptop resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/29 15:35:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 15:35:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/29 15:35:36 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/29 15:35:36 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/29 15:35:41 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Train Normal: 227398 | Train Fraud: 394


# 2️⃣ Prepare Three Different Training Strategies

## What This Section Does
Here we create three alternative training datasets so we can compare balancing strategies side by side.

## Why This Matters
Fraud detection is highly imbalanced. Different balancing techniques can affect recall, precision, and training speed in very different ways.

## The Three Strategies
- **Undersampling**: keep only a small sample of the normal class
- **Oversampling**: duplicate fraud cases so the fraud class becomes larger
- **Class Weights**: keep the raw data, but make fraud rows more important during training

## Code Breakdown
The code below:
- Calculates the fraction needed to undersample the normal class
- Randomly samples the normal rows
- Duplicates fraud rows for oversampling
- Creates a `class_weight` column for weighted training

## Key Idea
Each method tries to solve the same problem, but each one changes the data in a different way. That is why the next step is to train and compare the models carefully.

In [2]:
# --- STRATEGY 1: UNDERSAMPLING ---
fraction_under = fraud_count / normal_count
undersampled_normal = normal_train.sample(withReplacement=False, fraction=fraction_under, seed=42)
train_undersampled = fraud_train.unionByName(undersampled_normal)

# --- STRATEGY 2: OVERSAMPLING ---
# We duplicate the minority class (Fraud) to match the majority class
fraction_over = normal_count / fraud_count
oversampled_fraud = fraud_train.sample(withReplacement=True, fraction=fraction_over, seed=42)
train_oversampled = normal_train.unionByName(oversampled_fraud)

# --- STRATEGY 3: CLASS WEIGHTS ---
# We use the raw data, but add a 'weight' column. 
# Normal gets weight 1.0. Fraud gets a massive weight (e.g., 500.0)
balancing_ratio = normal_count / fraud_count
train_weighted = train_raw.withColumn("class_weight", when(col("Class") == 1, balancing_ratio).otherwise(1.0))

print("Datsets prepared successfully!")

Datsets prepared successfully!


# 3️⃣ Train the Models

## What This Section Does
Now we train four logistic regression models:
1. A default model with no balancing
2. A model trained on undersampled data
3. A model trained on oversampled data
4. A model trained with class weights

## Why This Matters
We want to see how balancing changes the model's behavior. Some methods may catch more fraud, but also create more false alarms.

## Code Breakdown
The code below:
- Creates a baseline `LogisticRegression` model
- Fits the model on the raw training data
- Fits the same type of model on the undersampled and oversampled datasets
- Creates a second logistic regression model that uses `weightCol="class_weight"`
- Trains the weighted model on the original data

## Important Detail
The weighted model keeps all the original data, which is useful because it does not throw away normal transactions or duplicate fraud rows. This often makes it a strong practical choice.

In [3]:
# 1. Default Model (No balancing)
print("Training Default Model...")
lr_default = LogisticRegression(featuresCol="features", labelCol="Class")
model_default = lr_default.fit(train_raw)

# 2. Undersampled Model
print("Training Undersampled Model...")
model_under = lr_default.fit(train_undersampled)

# 3. Oversampled Model
print("Training Oversampled Model...")
model_over = lr_default.fit(train_oversampled)

# 4. Weighted Model (Notice we add weightCol here!)
print("Training Weighted Model...")
lr_weighted = LogisticRegression(featuresCol="features", labelCol="Class", weightCol="class_weight")
model_weighted = lr_weighted.fit(train_weighted)

print("All models trained!")

Training Default Model...


Training Undersampled Model...
Training Oversampled Model...
Training Weighted Model...
All models trained!


# 4️⃣ Evaluate the Results

## What This Section Does
This section compares the four models on the same test set and prints precision and recall for each one.

## Why This Matters
Accuracy is not enough for fraud detection. A model can look good on paper while still missing most fraud cases.

## Code Breakdown
The code below:
- Runs each trained model on `test_data`
- Collects predictions and true labels
- Builds a confusion matrix using `MulticlassMetrics`
- Extracts true positives, false positives, false negatives, and true negatives
- Prints **precision** and **recall** for the fraud class

## How to Read the Metrics
- **Recall** tells us how much fraud we caught
- **Precision** tells us how many of the predicted fraud cases were actually fraud
- A good fraud model usually needs a balance between the two

In [4]:
def evaluate(model, name):
    print(f"\n{'='*40}")
    print(f"RESULTS FOR: {name}")
    print(f"{'='*40}")
    
    predictions = model.transform(test_data)
    preds_and_labels = predictions.select(['prediction', 'Class']).rdd.map(lambda r: (float(r.prediction), float(r.Class)))
    metrics = MulticlassMetrics(preds_and_labels)
    
    cm = metrics.confusionMatrix().toArray()
    TP = cm[1][1]
    FP = cm[0][1]
    FN = cm[1][0]
    TN = cm[0][0]
    
    precision = metrics.precision(1.0)
    recall = metrics.recall(1.0)
    
    print(f"Caught Fraud (TP): {TP}")
    print(f"Missed Fraud (FN): {FN}  <-- DANGER")
    print(f"False Alarms (FP): {FP}  <-- ANNOYED CUSTOMERS")
    print(f"Recall:    {recall * 100:.2f}% (How much total fraud did we catch?)")
    print(f"Precision: {precision * 100:.2f}% (When we say 'Fraud', how often are we right?)")

# Run the evaluations
evaluate(model_default, "1. DEFAULT MODEL (Raw Data)")
evaluate(model_under, "2. UNDERSAMPLING")
evaluate(model_over, "3. OVERSAMPLING")
evaluate(model_weighted, "4. CLASS WEIGHTS")


RESULTS FOR: 1. DEFAULT MODEL (Raw Data)


/home/serby/anomaly_detection_project/venv/lib/python3.12/site-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Caught Fraud (TP): 45.0
Missed Fraud (FN): 53.0  <-- DANGER
False Alarms (FP): 8.0  <-- ANNOYED CUSTOMERS
Recall:    45.92% (How much total fraud did we catch?)
Precision: 84.91% (When we say 'Fraud', how often are we right?)

RESULTS FOR: 2. UNDERSAMPLING


/home/serby/anomaly_detection_project/venv/lib/python3.12/site-packages/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Caught Fraud (TP): 88.0
Missed Fraud (FN): 10.0  <-- DANGER
False Alarms (FP): 1828.0  <-- ANNOYED CUSTOMERS
Recall:    89.80% (How much total fraud did we catch?)
Precision: 4.59% (When we say 'Fraud', how often are we right?)

RESULTS FOR: 3. OVERSAMPLING


Caught Fraud (TP): 86.0
Missed Fraud (FN): 12.0  <-- DANGER
False Alarms (FP): 1165.0  <-- ANNOYED CUSTOMERS
Recall:    87.76% (How much total fraud did we catch?)
Precision: 6.87% (When we say 'Fraud', how often are we right?)

RESULTS FOR: 4. CLASS WEIGHTS


Caught Fraud (TP): 86.0
Missed Fraud (FN): 12.0  <-- DANGER
False Alarms (FP): 1179.0  <-- ANNOYED CUSTOMERS
Recall:    87.76% (How much total fraud did we catch?)
Precision: 6.80% (When we say 'Fraud', how often are we right?)


# 5️⃣ How to Write the Conclusion

## What This Section Is For
This final note helps you interpret the results and turn them into a report conclusion.

## Main Takeaway
Each balancing method creates a different trade-off between catching fraud and avoiding false alarms.

## What You Will Usually See
- **Default model**: high precision, low recall
- **Undersampling**: high recall, but often many false positives
- **Oversampling**: better balance, but slower training and possible overfitting
- **Class weights**: often the most practical choice for large datasets

## What to Say in Your Report
Explain which method gave the best balance for your goal. For fraud detection, the most important question is usually: **How much fraud did the model catch without overwhelming users with false alarms?**

## Final Idea
There is no perfect method for every case. The best approach depends on whether your priority is catching as much fraud as possible, keeping false alarms low, or maintaining efficient training on big data.